# Prueba del endpoint `/llm/data-extraction`

Smoke test end-to-end del nuevo endpoint (ver `aymurai/api/endpoints/routers/llm/data_extraction/`):
llama al servidor real vía HTTP, igual que hace el frontend -- `/misc/document-extract`
para obtener el texto, y `/llm/data-extraction` para la extracción + la inferencia de
sector.

`nombre`, `cargo`, `tema` y `subtema` se devuelven tal cual los extrajo el LLM (son
campos de texto libre / dropdown en el front, sin listas de opciones generadas por el
backend). `sector` sí tiene candidatos rankeados:

- `candidatos_sector`: nombre/cargo se cruzan contra el organigrama y, a partir de los
  cargos candidatos, se resuelve el sector (tomando siempre la evidencia de cargo con
  peso completo y descontando la de nombre con `nombre_origen_weight`, porque el cargo
  es la señal más durable) -- solo se calcula cuando el LLM ya infirió `sector="GCBA"`.
  `sector_mode` elige cómo: `"csv"` (default) cruza contra `destinatario_por_sector.csv`
  tomando hasta `sector_top_k` matches por candidato; `"hierarchy"` lee el sector
  directo del organigrama (el primer nivel real bajo "Jefe de Gobierno"), sin tocar el CSV.

**Contenido de esta notebook:**
1. Descubrir documentos de prueba + helpers para llamar a la API.
2. Prueba sobre un solo documento (smoke test + demo de `candidatos_sector`).
3. Evaluación final: accuracy del pipeline completo contra un ground truth manual
   (`resolucion, destinatario_idx, nombre, cargo, sector_ground_truth`), barriendo
   hiperparámetros de búsqueda, con matriz de confusión y precision/recall/F1.


In [ ]:
import json
import mimetypes
import os
import time
from pathlib import Path
from collections.abc import Iterable

import pandas as pd
import requests
from tqdm import tqdm

API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
DOCUMENT_EXTRACT_ENDPOINT = "https://aymurai.collectiveai.io/api/misc/document-extract"
DATA_EXTRACTION_ENDPOINT = f"{API_BASE_URL}/llm/data-extraction"
DATA_ROOT = Path(
    os.getenv("DOCUMENT_DATA_ROOT", "../../../resources/data/restricted/defensoria/pdfs")
)
DOC_EXTENSIONS = {".pdf", ".docx"}
REQUEST_TIMEOUT = float(os.getenv("REQUEST_TIMEOUT", "120"))

print(f"Document extract endpoint: {DOCUMENT_EXTRACT_ENDPOINT}")
print(f"Data extraction endpoint:  {DATA_EXTRACTION_ENDPOINT}")
print(f"Documentos de prueba en:   {DATA_ROOT}")


In [ ]:
{
  "document": {
    "document": [
      "VISTO:",
      "La actuación no 3492/14 y el trámite no 16043/16 , iniciada/o de oficio por esta Defensoría del Pueblo y por una persona que solicitó reserva de su identidad, respectivamente, a fin de relevar las condiciones de los hoteles sitos en las calles República Bolivariana de Venezuela 4161 y Estados Unidos 1324/1326, ambos en esta Ciudad.",
      "Y CONSIDERANDO QUE:",
      "I. Hechos",
      "La actuación no 3492/14 , se inició de oficio por esta Defensoría del Pueblo atento haberse tomado conocimiento de las malas condiciones habitacionales del hotel sito en la calle República Bolivariana de Venezuela 4161 de esta Ciudad (fs. 1/3); y el trámite no 16043/16 , por una persona que solicitó se mantengan en reserva sus datos personales, la que denunció las malas condiciones del hotel sito en la calle Estados Unidos 1324/1326, también en esta Ciudad (fs. 1).",
      "A fs. 14, de la actuación no 3492/14 , la Dirección General de Habilitaciones y Permisos informó mediante la Providencia no PV-2014-10870000- -DGHP, que el establecimiento sito en la calle República Bolivariana de Venezuela 4161 de esta Ciudad, se encontraba habilitado para el rubro \"hotel sin servicio de comida\"; mientras que para el hotel ubicado en la calle Estados Unidos indicó, a través del Informe no IF-2017-17601432- -DGHP, que el mismo no poseía habilitación (fs. 39vta.).",
      "En ambos casos se solicitó la intervención de la Dirección General de Fiscalización y Control (DGFyC). De la documentación acompañada en respuesta, a fs. 19/22, de la actuación no",
      "Página 1 de 7",
      "3492/14 , surge que en el año 2011 fueron clausuradas once (11) habitaciones del establecimiento por encontrarse funcionando en exceso a la capacidad habilitada. En inspecciones efectuadas con posterioridad se constató que la medida de interdicción sobre dichas habitaciones no se estaba cumpliendo (fs. 55/70) razón por la cual se labró la correspondiente acta de intimación. Luego de ello, se realizó una nueva inspección en la que si bien se constató que las habitaciones clausuradas no se encontraban en uso, se observaron nuevas irregularidades por las que se cursó nueva intimación (fs. 145/148). En inspecciones posteriores se volvieron a labrar actas por haberse relevado nuevas faltas, por ejemplo: falta de certificados de fumigación y limpieza de tanques de agua, de plan y planos de evacuación, etc. (fs. 171); relevándose posteriormente que no se había dado cumplimiento, por lo cual se volvió a intimar (fs. 201vta./202).",
      "A fs. 248/251, se agregó un nuevo informe de la DGFyC, en el cual se consignó lo siguiente: \"... se constata que se trata de un establecimiento que se encuentra en adecuadas condiciones de higiene, aseo y regulares de mantenimiento edilicio...\" . Se informó, también que el tablero eléctrico se encontraba en adecuadas condiciones, y que había cantidad reglamentaria de luces de emergencia y de matafuegos. Con relación a la cantidad de habitaciones en uso, se indicó que coincidía con las habilitadas. Sin perjuicio de ello, se labraron actas de intimación por no exhibir sistema de autoprotección y por tener elevada carga de fuego en el sótano, entre otros puntos; cursándose posteriormente una nueva intimación por haberse relevado en nuevos operativos el incumplimiento (fs. 263/265 y 273 /274). A fs. 284/285, se informó luego de realizar una nueva verificación, que se había dado cumplimiento a la totalidad de las intimaciones.",
      "Finalmente, por medio del Informe no IF-2022-04716250-GCABA-AGC, se señaló que: \"... el local depósito PB se está utilizando como habitación (...) por lo q se procede a la clausura del local labrándose acta...\" . Asimismo, se labraron nuevas actas por tener dos (2) baños comunes sin agua, luz ni depósito de inodoro; tener grasa acumulada en la campana de la cocina; estar vencido el protocolo de puesta a tierra; falta de bandas de destaque en escaleras; falta de agua caliente en baños comunes de planta baja; etc. (fs. 295/297).",
      "Página 2 de 7",
      "Con relación al establecimiento sito en la calle Estados Unidos 1324/1326 de esta Ciudad (trámite no 16043/16 ), la DGFyC informó que el mismo había sido clausurado mediante Disposición no DI-2008/3514/DGFYC ampliada por su similar DI-2015/616/DGFYC (fs. 18 /19). A fs. 23vta./25, informó que no habían sido resueltas las faltas que originaron la interdicción (matafuegos descargado, falta de señalización de medios de salida, tenencia de cables expuestos, termotanque con ventilación antirreglamentaria, falta de carcaza de calefón dejando llama expuesta, grasa acumulada en la cocina, falta de habilitación, etc.). Posteriormente, se comunicó que se realizaron nuevos operativos, oportunidades en las cuales no había sido posible ingresar al establecimiento (fs. 43vta./46, 57/58 y 71/73). Sin embargo, personal del Hospital General de Agudos \"Dr. José María Ramos Mejía\", en respuesta a un oficio oportunamente librado en virtud de la situación sanitaria reinante en ese momento producto de la pandemia de público conocimiento, informó que se había relevado que en el lugar funcionaba un hotel, con tres (3) pisos y aproximadamente quince (15) habitaciones por piso (fs. 102).",
      "A fs. 109/118, mediante Informe no IF-2021-25717535-GCABA-AGC, la Administración consignó a raíz de un nuevo relevamiento que: \"... Se trata de un establecimiento en buenas condiciones de conservación edilicia y de higiene...\" ; y que se había procedido a la clausura del mismo por no contar con habilitación. Asimismo, se destacó que la clausura impuesta por Disposición no DI-2008/3514/DGFYC ampliada por su similar DI-2015/616/DGFYC, se encontraba vigente, pero que en la actualidad había un nuevo explotador comercial.",
      "II. Normativa aplicable",
      "[1] [2] El art. 23 de la Ley no 6101 (según texto consolidado por Ley no 6347 ) (Ley Marco de Regulación de Actividades Económicas en la Ciudad Autónoma de Buenos Aires) establece que: \"... La función de evaluación y comprobación de conformidad con el ordenamiento jurídico del ejercicio de las actividades económicas es realizada por los inspectores. La autoridad de aplicación establecerá el órgano fiscalizador que planificará y coordinará las actividades de los inspectores\".",
      "Página 3 de 7",
      "Asimismo, en su art. 26 dentro de las facultades del personal inspectivo, estipula que el mismo está facultado \"... para realizar las siguientes acciones: a. Verificar y constatar infracciones mediante el acta correspondiente en los términos del artículo 3° de la Ley 1217. b. Intimar la subsanación de defectos en el ejercicio de la actividad en un plazo razonable. Dicha subsanación deberá ser demostrada por el ciudadano mediante la presentación de la documentación correspondiente en la sede del órgano fiscalizador en el plazo intimado. c. Clausurar de manera inmediata y preventiva el lugar en infracción en los términos del artículo 7° de la Ley 1217...\" .",
      "Por su parte, con relación a las habilitaciones, el art. 21 de la normativa citada expresa que: \"... La autoridad de aplicación podrá por acto fundado revocar las autorizaciones otorgadas bajo las siguientes causales: a) Cuando se encuentre afectada o en peligro en forma inmediata la salubridad y/o seguridad pública. b) Cuando se constatare el falseamiento de la declaración responsable o los instrumentos acompañados con ella. c) Cuando se compruebe por segunda vez la violación de la clausura. d) Cuando se constatare por segunda vez la desvirtuación del rubro y/o uso objeto de la autorización de la actividad económica\".",
      "[3] La Resolución no 27.881/1973 , dispone que cuando corresponda clausurar y/o cancelar habilitaciones de hoteles, residenciales o pensiones, arbitrará las medidas necesarias a efectos de que los/as ocupantes puedan permanecer en los mismos, a pesar de la interdicción.",
      "[4] A través de la Ley no 2624 (según texto consolidado por Ley no 6347) de la Ciudad Autónoma de Buenos Aires, se crea \"... la Agencia Gubernamental de Control, entidad autárquica en el ámbito del Ministerio de Justicia y Seguridad de la Ciudad...\" (art. 1o). A través de la Dirección General de Fiscalización y Control (art. 4o) ejerce la competencia de control del Código de Habilitaciones y Verificaciones (art. 5o inc. a). Asimismo, a través de la Dirección General de Fiscalización y Control de Obras ejerce el contralor del Código de Edificación y de sus normas complementarias (art. 5o inc. c). El art. 2o, establece que ejerce el contralor, fiscalización y regulación, con facultades de recurrir a la fuerza pública. Se indica también, que podrá aplicar multas y sanciones.",
      "Página 4 de 7",
      "Por otra parte, la violación de la clausura constituye una contravención, de acuerdo al art. 76 del Código Contravencional de esta Ciudad, que estipula lo siguiente: \"a) Violar Clausura. El titular del establecimiento donde se viole una clausura impuesta por autoridad judicial o administrativa, es sancionado/a con treinta mil pesos ($30.000) a sesenta mil pesos ($60.000) de multa o cinco (5) a veinte (20) días de arresto (...) b) El que incumple una sanción sustitutiva o accesoria impuesta por infracción al régimen de faltas por sentencia firme de autoridad judicial es sancionado/a con cuarenta mil pesos ($40.000) a setenta mil pesos ($70.000) de multa o cinco (5) a quince (15) días de arresto\".",
      "III. Conclusión",
      "Del análisis de los presentes actuados surge que las irregularidades que dieron origen a los mismos (deficiencias en los servicios de alojamiento) se mantenían en el tiempo. Sin perjuicio de ello, la Dirección General de Fiscalización y Control ha tomado la intervención del caso, ya que se realizaron sucesivas inspecciones, labrándose actas en caso de corresponder y efectuando posteriormente el control de las mismas. Sin embargo, y teniendo en cuenta que en cada operativo se relevaron nuevas faltas, corresponde cursar la recomendación del caso a fin de que se mantengan en observación los hoteles denunciados.",
      "POR TODO ELLO:",
      "LA DEFENSORA DEL PUEBLO",
      "DE LA CIUDAD AUTÓNOMA DE BUENOS AIRES",
      "R E S U E L V E :",
      "1) Recomendar al Director General de Fiscalización y Control de la Agencia Gubernamental de Control del Gobierno de la Ciudad Autónoma de Buenos Aires, señor Santiago Pedro",
      "Página 5 de 7",
      "Delucchi, tenga a bien, mantener en observación mediante inspecciones periódicas, los establecimientos hoteleros sitos en las calles República Bolivariana de Venezuela 4161 y Estados Unidos 1324/1326, ambos en esta Ciudad; e informar a esta Defensoría del Pueblo los resultados de lo actuado.",
      "[5] 2) Fijar en treinta (30) días el plazo previsto en el art. 36 de la Ley no 3 (según texto [6] consolidado por Ley no 6347) de la Ciudad Autónoma de Buenos Aires .",
      "3) Registrar, notificar, reservar en la Coordinación Operativa para su seguimiento y oportunamente archivar.",
      "Código 401",
      "da/dl/COCC/CEDUEPMA",
      "co-abda/COCF/CEAL",
      "as/ea/SOADA",
      "gv./MAER/COMESA",
      "Notas",
      "1. ^ Ley no 6101, sancionada el día 6 de diciembre de 2018, promulgada con fecha 26 de diciembre de 2018, y publicada en el Boletín Oficial no 5.526 del 27 de diciembre de 2018.",
      "2. ^ Ley no 6347, sancionada el día 12 de noviembre de 2020, promulgada con fecha 27 de noviembre de 2020, y publicada en el Boletín Oficial no 6.009 del 1o de diciembre de 2020.",
      "3. ^ Resolución no 27.881/1973, sancionada el día 24 de julio de 1973, promulgada con fecha 12 de septiembre de 1973, y publicada en el Boletín Municipal del 14 de septiembre de 1973.",
      "4. ^ Ley no 2624, sancionada el día 13 de diciembre de 2007, promulgada con fecha 28 de diciembre de 2007, y publicada en el Boletín Oficial no 2.843 del 4 de enero de 2008.",
      "Página 6 de 7",
      "5. ^ Ley no 3 de la Ciudad Autónoma de Buenos Aires, sancionada el día 3 de febrero de 1998 y publicada en el Boletín Oficial no 394 de fecha 27 de febrero de 1998.",
      "6. ^ Ley no 3, art. 36: \"Con motivo de sus investigaciones, el Defensor o Defensora del Pueblo puede formular advertencias, recomendaciones, recordatorios de los deberes de los funcionarios, y propuestas para la adopción de nuevas medidas. Las recomendaciones no son vinculantes, pero si dentro del plazo fijado la autoridad administrativa afectada no produce una medida adecuada, o no informa de las razones que estime para no adoptarla, el Defensor o Defensora del Pueblo puede poner en conocimiento del ministro o secretario del área, o de la máxima autoridad de la entidad involucrada, los antecedentes del asunto y las recomendaciones propuestas. Si tampoco así obtiene una justificación adecuada, debe incluir tal asunto en su informe anual o especial a la Legislatura, con mención de los nombres de las autoridades o funcionarios que hayan adoptado tal actitud\".",
      "Página 7 de 7"
    ],
    "document_id": "06cb0508-1f4a-57c4-b5b4-dbf5c0770d42",
    "header": null,
    "footer": null,
    "page_count": 7
  },
  "model": null,
  "options": null
}

## Descubrir documentos de prueba

In [ ]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Directory '{DATA_ROOT}' not found. Update DATA_ROOT before continuing."
    )


def discover_documents(root: Path, extensions: Iterable[str]) -> list[Path]:
    """Recursively find the documents to test under `root`.

    Args:
        root (Path): Root folder to search (recursively).
        extensions (Iterable[str]): Extensions to include (with or without leading dot).

    Returns:
        list[Path]: Matching paths, sorted alphabetically.
    """
    extensions = {ext.lower() for ext in extensions}
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    )


documents = discover_documents(DATA_ROOT, DOC_EXTENSIONS)
print(f"Discovered {len(documents)} documents.")
documents[:5]

## Helpers para llamar a la API

In [ ]:
def call_extraction_api(
    session: requests.Session, file_path: Path
) -> dict[str, object]:
    """
    POST a file to /misc/document-extract and return a status-wrapped payload.

    Args:
        session (requests.Session): HTTP session reused across calls.
        file_path (Path): Local path of the file to upload.

    Returns:
        dict[str, object]: {"path", "status" ("success"/"failure"), "status_code",
        "elapsed_s", "detail"} -- if status="success", detail holds
        {"document_id", "document"}; otherwise it holds the error detail.
    """
    payload: dict[str, object] = {
        "path": str(file_path),
        "status": "failure",
        "status_code": None,
        "elapsed_s": None,
        "detail": None,
    }

    if not file_path.exists():
        payload["detail"] = "File does not exist"
        return payload

    mime_type = mimetypes.guess_type(file_path.name)[0] or "application/octet-stream"
    files = {"file": (file_path.name, file_path.open("rb"), mime_type)}

    try:
        start = time.perf_counter()
        response = session.post(DOCUMENT_EXTRACT_ENDPOINT, files=files, timeout=REQUEST_TIMEOUT)
        elapsed = time.perf_counter() - start
    except requests.RequestException as exc:
        payload["detail"] = f"Request failed: {exc}"
        return payload
    finally:
        files["file"][1].close()

    payload["status_code"] = response.status_code
    payload["elapsed_s"] = elapsed

    try:
        response_body = response.json()
    except ValueError:
        response_body = {"raw": response.text[:500]}

    if response.ok:
        payload["status"] = "success"
        payload["detail"] = {
            "document_id": response_body.get("document_id"),
            "document": response_body.get("document", []),
        }
    else:
        payload["detail"] = response_body

    return payload


def call_data_extraction(
    session: requests.Session, document: dict, use_cache: bool = True, **overrides
) -> dict:
    """POST a Document payload to /llm/data-extraction and return the result.

    Args:
        session (requests.Session): HTTP session reused across calls.
        document (dict): Document payload ({"document", "document_id"}), as
            returned by call_extraction_api.
        use_cache (bool): Goes as a *query param*, not part of the JSON body
            (same convention as the anonymizer endpoint) -- True (default)
            returns/stores a persisted result for this document_id; False
            always re-runs the LLM/organigram pipeline and skips the DB
            entirely (nothing read, nothing written).
        **overrides: Optional request body fields (model, search_backend,
            hybrid_weight, top_k, sector_mode, sector_top_k,
            nombre_origen_weight, max_retries, options).

    Returns:
        dict: The DataExtractionResult returned by the endpoint, as a dict.
    """
    payload = {"document": document, **overrides}
    response = session.post(
        DATA_EXTRACTION_ENDPOINT,
        json=payload,
        params={"use_cache": use_cache},
        timeout=REQUEST_TIMEOUT,
    )
    if not response.ok:
        print(f"Error {response.status_code}: {response.text[:500]}")
    response.raise_for_status()
    return response.json()


## Prueba sobre un solo documento

In [ ]:
session = requests.Session()

for doc_path in documents:
    if doc_path.name == "65-23cc.pdf":
        break
    else:
        doc_path = documents[5]
print(f"Documento: {doc_path.name}")

extracted = call_extraction_api(session, doc_path)
if extracted["status"] != "success":
    raise RuntimeError(f"Extraction failed for {doc_path.name}: {extracted['detail']}")

document = extracted["detail"]
print(f"document_id: {document['document_id']}")
print(f"Párrafos extraídos: {len(document['document'])}")


In [ ]:
result_default = call_data_extraction(session, document, use_cache=True, top_k=10, nombre_origen_weight=0.3, hybrid_weight=0.5, sector_top_k=5, sector_mode='csv')
print(json.dumps(result_default, indent=2, ensure_ascii=False))

### Sector: el nuevo dropdown

`candidatos_sector` (por destinatario) es el campo que reemplaza a los viejos
`candidatos_nombre`/`candidatos_cargo`. `tema`/`subtema` los maneja el front
directamente contra su propia taxonomía -- el endpoint no genera listas de opciones
para ellos.


In [ ]:
print("=== sector: valor del LLM + candidatos rankeados ===")
for idx, dest in enumerate(result_default["destinatarios"]):
    print(
        f"Destinatario {idx}: nombre={dest['nombre']!r} cargo={dest['cargo']!r} "
        f"sector (LLM)={dest['sector']!r}"
    )
    for candidato in dest["candidatos_sector"]:
        print(
            f"    -> {candidato['sector']} (score={candidato['score']:.3f}) "
            f"<- organigrama[{candidato['origen_campo']}]: "
            f"nombre={candidato['origen_nombre']!r} cargo={candidato['origen_cargo']!r} "
            f"(score={candidato['origen_score']:.3f})"
        )

print("\n=== tema / subtema (LLM, sin dropdown propio del backend) ===")
print(f"tema: {result_default['tema']!r}")
print(f"subtema: {result_default['subtema']!r}")


## Evaluación final: accuracy contra ground truth

Compara el pipeline completo contra un CSV de ground truth con columnas
`resolucion, destinatario_idx, nombre, cargo, sector_ground_truth` (una fila por
destinatario esperado). `resolucion` debe matchear el nombre de archivo del PDF
(`doc_path.name`). `sector_ground_truth` sirve doble propósito, igual que
`destinatario_por_sector.csv`: para destinatarios no-GCBA es una de
`"Empresa"`/`"Organismos Nacionales"`/`"Obra Social / Prepaga"`; para destinatarios
GCBA es el sector específico (ej. `"Ministerio de Educación"`, `"AGC"`). Las filas
donde `sector_ground_truth` valga `"no definido"` se excluyen de toda comparación de
sector (macro y fina) -- son casos que todavía no se resolvieron a mano; sí cuentan
para el chequeo de cantidad de destinatarios, que no depende del sector.

La evaluación tiene dos niveles:

1. **Extracción (nivel macro)**: por cada `resolucion`, ¿el pipeline detectó la
   cantidad correcta de destinatarios? Por cada destinatario, ¿el campo `sector`
   que devuelve el LLM (GCBA/Empresa/Organismos Nacionales/Obra Social) coincide
   con la categoría macro derivada de `sector_ground_truth`? Esto usa una sola
   corrida del LLM por documento (config fija) -- no depende de `hybrid_weight`
   ni `nombre_origen_weight`, que no tocan la extracción.
2. **Sector fino (solo destinatarios GCBA)**: accuracy top-1/top-3/top-5 de
   `candidatos_sector`, comparando `sector_mode="csv"` vs `"hierarchy"`, barriendo
   `hybrid_weight` y `nombre_origen_weight` en `[0, 0.25, 0.5, 0.75, 1]` (5x5 = 25
   combinaciones). El nombre/cargo usados como insumo son los que **extrajo
   nuestro propio pipeline** (no los de ground truth) -- es una evaluación
   end-to-end, no solo del paso de matching aislado.

Como en las secciones anteriores: el LLM se llama **una sola vez por documento**
(no depende de estos parámetros), y el barrido completo se recalcula localmente
importando `organigram_matching`/`sector_matching` directamente.


In [ ]:
import sys

os.environ["RESOURCES_BASEPATH"] = str(Path("../../../resources").resolve())
for module_name in list(sys.modules):
    if module_name == "aymurai.settings" or module_name.startswith("aymurai.api"):
        del sys.modules[module_name]

from aymurai.api.endpoints.routers.llm.data_extraction import organigram_matching as om
from aymurai.api.endpoints.routers.llm.data_extraction import sector_matching as sm

GROUND_TRUTH_CSV_PATH = Path(
    "../../../resources/data/restricted/defensoria/documento_sector_ground_truth.csv"
).resolve()

if not GROUND_TRUTH_CSV_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el ground truth en {GROUND_TRUTH_CSV_PATH} -- "
        "actualizá GROUND_TRUTH_CSV_PATH con la ruta real."
    )

ground_truth_df = pd.read_csv(GROUND_TRUTH_CSV_PATH)
columnas_esperadas = {"resolución", "destinatario_idx", "nombre", "cargo", "sector_ground_truth"}
faltantes = columnas_esperadas - set(ground_truth_df.columns)
if faltantes:
    raise ValueError(f"Al ground truth le faltan columnas: {faltantes}")

print(f"Ground truth: {len(ground_truth_df)} filas, {ground_truth_df['resolución'].nunique()} documentos")

doc_by_name = {p.name: p for p in documents}
sin_documento = sorted(set(ground_truth_df["resolución"]) - set(doc_by_name))
if sin_documento:
    print(f"AVISO: {len(sin_documento)} 'resolución' del ground truth no matchean ningún archivo en DATA_ROOT:")
    for nombre_archivo in sin_documento:
        print(f"  {nombre_archivo}")


### Paso 1: extraer cada documento una sola vez

Config fija (no es parte del barrido): `search_backend="hybrid"`, `hybrid_weight=0.5`.
Estos valores solo importan para lo que se recalcula localmente después -- acá el
LLM ya devolvió `destinatarios`/`sector` y no hace falta llamarlo de nuevo.

El resultado se cachea en `EXTRACTION_CACHE_PATH` (JSON): si volvés a correr esta
celda más adelante (kernel reiniciado, otro día, ground truth con filas nuevas), los
documentos ya extraídos **no se vuelven a mandar al LLM** -- solo se extraen los que
todavía no están en el cache. Si querés forzar una re-extracción completa, borrá ese
archivo (o la entrada puntual del documento que quieras recalcular).


In [ ]:
EXTRACTION_CACHE_PATH = Path(
    "../../../resources/data/restricted/defensoria/results/extraction_cache_eval.json"
).resolve()

if EXTRACTION_CACHE_PATH.exists():
    extraction_by_resolucion = json.loads(EXTRACTION_CACHE_PATH.read_text(encoding="utf-8"))
    print(f"Cache cargado: {len(extraction_by_resolucion)} documentos ya extraídos previamente")
else:
    extraction_by_resolucion = {}

resoluciones_a_procesar = sorted(
    (set(ground_truth_df["resolución"]) & set(doc_by_name)) - set(extraction_by_resolucion)
)
print(f"Documentos nuevos a extraer: {len(resoluciones_a_procesar)}")

errores_extraccion = []

for resolucion in tqdm(resoluciones_a_procesar, desc="Extrayendo documentos"):
    doc_path = doc_by_name[resolucion]
    try:
        extracted = call_extraction_api(session, doc_path)
        if extracted["status"] != "success":
            errores_extraccion.append({"resolucion": resolucion, "error": extracted["detail"]})
            continue

        document = extracted["detail"]
        if not document["document"]:
            errores_extraccion.append({"resolucion": resolucion, "error": "documento vacío"})
            continue

        result = call_data_extraction(session, document, search_backend="hybrid", hybrid_weight=0.5)
        extraction_by_resolucion[resolucion] = result
    except Exception as exc:
        errores_extraccion.append({"resolucion": resolucion, "error": str(exc)})

EXTRACTION_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
EXTRACTION_CACHE_PATH.write_text(
    json.dumps(extraction_by_resolucion, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(f"Documentos extraídos en total (cache + nuevos): {len(extraction_by_resolucion)}")
print(f"Errores en esta corrida: {len(errores_extraccion)}")
for e in errores_extraccion:
    print(" ", e)


### Punto 1: cantidad de destinatarios + sector macro

Por documento: ¿la cantidad de destinatarios extraídos coincide con la del ground
truth? Por destinatario (matcheado por `destinatario_idx`): ¿la categoría macro de
`sector` (GCBA vs Empresa vs Organismos Nacionales vs Obra Social/Prepaga) coincide
con la derivada de `sector_ground_truth`?


In [ ]:
SECTORES_NO_GCBA = {"Empresa", "Organismos Nacionales", "Obra Social / Prepaga"}
NO_DEFINIDO = "no definido"


def es_no_definido(valor: str) -> bool:
    """True si sector_ground_truth todavía no está determinado (caso a resolver a futuro)."""
    return isinstance(valor, str) and valor.strip().lower() == NO_DEFINIDO


def macro_sector(sector_ground_truth: str) -> str:
    """Derive the macro sector (GCBA/Empresa/Organismos Nacionales/Obra Social)
    from a (possibly granular) sector_ground_truth value."""
    return sector_ground_truth if sector_ground_truth in SECTORES_NO_GCBA else "GCBA"


conteo_rows = []
sector_macro_rows = []

for resolucion, grupo in ground_truth_df.groupby("resolución"):
    result = extraction_by_resolucion.get(resolucion)
    if result is None:
        continue

    destinatarios_extraidos = result.get("destinatarios", [])
    conteo_rows.append({
        "resolución": resolucion,
        "n_esperado": len(grupo),
        "n_extraido": len(destinatarios_extraidos),
        "cuenta_correcta": len(grupo) == len(destinatarios_extraidos),
    })

    for _, fila in grupo.iterrows():
        if es_no_definido(fila["sector_ground_truth"]):
            continue

        idx = int(fila["destinatario_idx"])
        macro_esperado = macro_sector(fila["sector_ground_truth"])
        if idx < len(destinatarios_extraidos):
            macro_obtenido = destinatarios_extraidos[idx].get("sector")
        else:
            macro_obtenido = None

        sector_macro_rows.append({
            "resolución": resolucion,
            "destinatario_idx": idx,
            "macro_esperado": macro_esperado,
            "macro_obtenido": macro_obtenido,
            "correcto": macro_obtenido == macro_esperado,
        })

conteo_df = pd.DataFrame(conteo_rows)
sector_macro_df = pd.DataFrame(sector_macro_rows)

print(f"Documentos con cantidad de destinatarios correcta: {conteo_df['cuenta_correcta'].mean():.1%}")
display(conteo_df[~conteo_df["cuenta_correcta"]])

print(f"\nDestinatarios evaluados (excluyendo '{NO_DEFINIDO}'): {len(sector_macro_df)}")
print(f"Destinatarios con sector macro correcto: {sector_macro_df['correcto'].mean():.1%}")
display(sector_macro_df[~sector_macro_df["correcto"]])


### Puntos 2-4: accuracy de sector fino (`csv` vs `hierarchy`)

Solo para destinatarios donde `sector_ground_truth` es GCBA (no una de las 3
categorías macro no-GCBA). Se usan el `nombre`/`cargo` que **extrajo nuestro propio
pipeline** para ese destinatario (no los del ground truth) -- evalúa el pipeline
completo, no solo el paso de matching de forma aislada.

Por cada destinatario: para cada `hybrid_weight` en la grilla, se calculan los
candidatos de organigrama **una sola vez** (hybrid_weight sí afecta este paso, así
que no se puede compartir entre valores de hybrid_weight); para cada
`nombre_origen_weight` en la grilla, se recalculan `candidatos_sector` en ambos
modos (`csv`/`hierarchy`) a partir de esos mismos candidatos, y se chequea si
`sector_ground_truth` aparece en el top-1/top-3/top-5.


In [ ]:
HYBRID_WEIGHTS_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]
NOMBRE_ORIGEN_WEIGHTS_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]
SECTOR_TOP_K_GRID = [1, 3, 5, 10]  # solo aplica a sector_mode="csv" -- hierarchy no lo usa
ORGANIGRAM_TOP_K_EVAL = 10  # fijo -- no es parte del barrido (ver nota mas arriba)

fine_accuracy_rows = []

destinatarios_gcba_eval = []
for _, fila in ground_truth_df.iterrows():
    if es_no_definido(fila["sector_ground_truth"]):
        continue  # todavia no sabemos el sector real de este destinatario
    if macro_sector(fila["sector_ground_truth"]) != "GCBA":
        continue
    resolucion = fila["resolución"]
    idx = int(fila["destinatario_idx"])
    result = extraction_by_resolucion.get(resolucion)
    if result is None:
        continue
    destinatarios_extraidos = result.get("destinatarios", [])
    if idx >= len(destinatarios_extraidos):
        continue  # el pipeline no detecto este destinatario -- ya se cuenta como error en el punto 1
    dest = destinatarios_extraidos[idx]
    if not (dest.get("nombre") or dest.get("cargo")):
        continue
    destinatarios_gcba_eval.append((resolucion, idx, dest, fila["sector_ground_truth"]))

print(f"Destinatarios GCBA evaluables (extraidos + con nombre/cargo): {len(destinatarios_gcba_eval)}")

for resolucion, idx, dest, sector_esperado in tqdm(destinatarios_gcba_eval, desc="Evaluando sector fino"):
    for hybrid_weight in HYBRID_WEIGHTS_GRID:
        organigram_matches = om.search_candidates(
            nombre=dest["nombre"],
            cargo=dest["cargo"],
            sector=dest["sector"],
            backend="hybrid",
            top_k=ORGANIGRAM_TOP_K_EVAL,
            hybrid_weight=hybrid_weight,
        )
        candidates_csv = [
            (field, c.nombre, c.cargo, c.score)
            for field, field_candidates in organigram_matches.items()
            for c in field_candidates
        ]
        candidates_hierarchy = [
            (field, c.nombre, c.cargo, c.score, c.ruta_cargos)
            for field, field_candidates in organigram_matches.items()
            for c in field_candidates
        ]

        for nombre_origen_weight in NOMBRE_ORIGEN_WEIGHTS_GRID:
            # hierarchy no depende de sector_top_k -- se calcula una sola vez por
            # (hybrid_weight, nombre_origen_weight), no una vez por sector_top_k.
            resultado_hierarchy = sm.search_sector_candidates_from_hierarchy(
                candidates_hierarchy, backend="hybrid", nombre_origen_weight=nombre_origen_weight,
            )
            sectores_hierarchy = [c.sector for c in resultado_hierarchy]
            for top_k in (1, 3, 5):
                fine_accuracy_rows.append({
                    "resolución": resolucion,
                    "destinatario_idx": idx,
                    "hybrid_weight": hybrid_weight,
                    "nombre_origen_weight": nombre_origen_weight,
                    "sector_top_k": None,
                    "metodo": "hierarchy",
                    "top_k": top_k,
                    "acierto": sector_esperado in sectores_hierarchy[:top_k],
                })

            # csv si depende de sector_top_k -- se barre por separado.
            for sector_top_k in SECTOR_TOP_K_GRID:
                resultado_csv = sm.search_sector_candidates(
                    candidates_csv,
                    backend="hybrid",
                    hybrid_weight=hybrid_weight,
                    sector_top_k=sector_top_k,
                    nombre_origen_weight=nombre_origen_weight,
                )
                sectores_csv = [c.sector for c in resultado_csv]
                for top_k in (1, 3, 5):
                    fine_accuracy_rows.append({
                        "resolución": resolucion,
                        "destinatario_idx": idx,
                        "hybrid_weight": hybrid_weight,
                        "nombre_origen_weight": nombre_origen_weight,
                        "sector_top_k": sector_top_k,
                        "metodo": "csv",
                        "top_k": top_k,
                        "acierto": sector_esperado in sectores_csv[:top_k],
                    })

fine_accuracy_df = pd.DataFrame(fine_accuracy_rows)
print(f"Filas de evaluacion: {len(fine_accuracy_df)}")
fine_accuracy_df.head()


In [ ]:
print("=== Ranking de configuraciones (accuracy top-1), mejor a peor ===")
top1_df = fine_accuracy_df[fine_accuracy_df["top_k"] == 1]
ranking = (
    top1_df
    .groupby(["metodo", "hybrid_weight", "nombre_origen_weight", "sector_top_k"], dropna=False)["acierto"]
    .mean()
    .reset_index()
    .rename(columns={"acierto": "accuracy"})
    .sort_values("accuracy", ascending=False)
)
display(ranking.head(15))

print("\n=== Mejor configuracion por metodo (top-1) ===")
mejores = {}
for metodo in ("csv", "hierarchy"):
    mejor = ranking[ranking["metodo"] == metodo].iloc[0]
    mejores[metodo] = mejor
    print(f"{metodo}: hybrid_weight={mejor['hybrid_weight']}, "
          f"nombre_origen_weight={mejor['nombre_origen_weight']}, "
          f"sector_top_k={mejor['sector_top_k']} -> accuracy={mejor['accuracy']:.1%}")

print("\n=== Esa misma configuracion, accuracy por top_k ===")
for metodo, mejor in mejores.items():
    filtro = (
        (fine_accuracy_df["metodo"] == metodo)
        & (fine_accuracy_df["hybrid_weight"] == mejor["hybrid_weight"])
        & (fine_accuracy_df["nombre_origen_weight"] == mejor["nombre_origen_weight"])
    )
    if metodo == "csv":
        filtro &= fine_accuracy_df["sector_top_k"] == mejor["sector_top_k"]
    print(f"{metodo}:")
    for top_k in (1, 3, 5):
        acc = fine_accuracy_df[filtro & (fine_accuracy_df["top_k"] == top_k)]["acierto"].mean()
        print(f"  top-{top_k}: {acc:.1%}")

### Métricas de clasificación: matriz de confusión y precision/recall/F1

El accuracy simple no dice **dónde** falla el pipeline -- si confunde sistemáticamente
dos sectores puntuales, o si le va mal solo con los sectores que tienen pocos ejemplos.
Esto agrega, con métricas estándar de clasificación (calculadas a mano con pandas, sin
sumar `sklearn` como dependencia nueva):

- **Matriz de confusión**: filas = sector esperado, columnas = sector que dio el
  pipeline. Los valores fuera de la diagonal son las confusiones puntuales.
- **Precision/recall/F1 por sector**: `precision` = de las veces que el pipeline dijo
  "sector X", ¿en qué fracción tenía razón?; `recall` = de las veces que el sector
  real era X, ¿en qué fracción el pipeline lo detectó?; `F1` combina ambas.
  `soporte` = cuántos casos reales tiene ese sector en el ground truth (los sectores
  con soporte bajo son los menos confiables de generalizar).

Primero para el sector macro (Punto 1); después para el sector fino, en la mejor
configuración encontrada para cada método (`csv`/`hierarchy`).


In [ ]:
def metricas_por_clase(esperado: pd.Series, obtenido: pd.Series) -> pd.DataFrame:
    """Precision/recall/F1/soporte por clase, calculado a mano (sin sklearn).

    Args:
        esperado (pd.Series): Etiqueta real, una por caso.
        obtenido (pd.Series): Etiqueta que dio el pipeline, misma longitud/orden.

    Returns:
        pd.DataFrame: Indexado por clase (solo las que aparecen en `esperado`),
        con columnas soporte/precision/recall/f1, ordenado por soporte descendente.
    """
    filas = []
    for clase in sorted(esperado.dropna().unique()):
        tp = ((esperado == clase) & (obtenido == clase)).sum()
        fp = ((esperado != clase) & (obtenido == clase)).sum()
        fn = ((esperado == clase) & (obtenido != clase)).sum()
        soporte = (esperado == clase).sum()
        precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
        recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
        f1 = (
            2 * precision * recall / (precision + recall)
            if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
            else float("nan")
        )
        filas.append({"sector": clase, "soporte": soporte, "precision": precision, "recall": recall, "f1": f1})
    return pd.DataFrame(filas).set_index("sector").sort_values("soporte", ascending=False)


print("=== Punto 1 (sector macro): matriz de confusion ===")
confusion_macro = pd.crosstab(
    sector_macro_df["macro_esperado"], sector_macro_df["macro_obtenido"], dropna=False
)
display(confusion_macro)

print("\n=== Punto 1 (sector macro): precision / recall / F1 ===")
display(metricas_por_clase(sector_macro_df["macro_esperado"], sector_macro_df["macro_obtenido"]))


### Métricas del sector fino, en la mejor configuración de cada método

La grilla de arriba solo guardó acierto/desacierto (top-1/3/5), no el sector que
realmente predijo el pipeline -- para armar la matriz de confusión hace falta
recuperar esa predicción. Se recalcula, para la mejor config de cada método
(`mejores`, ya calculado arriba), solo el top-1 real -- gracias a la cache de
embeddings esto es rápido (son los mismos `hybrid_weight` que ya se corrieron en la
grilla).


In [ ]:
def predicciones_top1(config: pd.Series, metodo: str) -> pd.DataFrame:
    """Recompute the top-1 predicted sector (not just hit/miss) for one config.

    Args:
        config (pd.Series): Row from `ranking` with hybrid_weight,
            nombre_origen_weight, and (for "csv") sector_top_k.
        metodo (str): "csv" or "hierarchy".

    Returns:
        pd.DataFrame: One row per destinatario evaluable, con
        resolucion/destinatario_idx/sector_esperado/sector_predicho.
    """
    filas = []
    for resolucion, idx, dest, sector_esperado in destinatarios_gcba_eval:
        organigram_matches = om.search_candidates(
            nombre=dest["nombre"],
            cargo=dest["cargo"],
            sector=dest["sector"],
            backend="hybrid",
            top_k=ORGANIGRAM_TOP_K_EVAL,
            hybrid_weight=config["hybrid_weight"],
        )
        if metodo == "csv":
            candidates = [
                (field, c.nombre, c.cargo, c.score)
                for field, field_candidates in organigram_matches.items()
                for c in field_candidates
            ]
            resultado = sm.search_sector_candidates(
                candidates,
                backend="hybrid",
                hybrid_weight=config["hybrid_weight"],
                sector_top_k=int(config["sector_top_k"]),
                nombre_origen_weight=config["nombre_origen_weight"],
            )
        else:
            candidates = [
                (field, c.nombre, c.cargo, c.score, c.ruta_cargos)
                for field, field_candidates in organigram_matches.items()
                for c in field_candidates
            ]
            resultado = sm.search_sector_candidates_from_hierarchy(
                candidates, backend="hybrid", nombre_origen_weight=config["nombre_origen_weight"],
            )

        sector_predicho = resultado[0].sector if resultado else None
        filas.append({
            "resolución": resolucion,
            "destinatario_idx": idx,
            "sector_esperado": sector_esperado,
            "sector_predicho": sector_predicho,
        })
    return pd.DataFrame(filas)


predicciones_por_metodo = {}
for metodo, mejor in mejores.items():
    print(f"Recalculando predicciones top-1 para la mejor config de {metodo}...")
    predicciones_por_metodo[metodo] = predicciones_top1(mejor, metodo)

resumen_f1 = []
for metodo, preds in predicciones_por_metodo.items():
    print(f"\n=== {metodo}: matriz de confusion (top-1), mejor config ===")
    confusion = pd.crosstab(preds["sector_esperado"], preds["sector_predicho"], dropna=False)
    display(confusion)

    print(f"\n=== {metodo}: precision / recall / F1 por sector, mejor config ===")
    metricas = metricas_por_clase(preds["sector_esperado"], preds["sector_predicho"])
    display(metricas)

    resumen_f1.append({
        "metodo": metodo,
        "f1_macro": metricas["f1"].mean(),
        "f1_ponderado_por_soporte": (metricas["f1"] * metricas["soporte"]).sum() / metricas["soporte"].sum(),
    })

print("\n=== Resumen: F1 macro vs F1 ponderado, csv vs hierarchy ===")
display(pd.DataFrame(resumen_f1).set_index("metodo"))


### `clean_cargo`: cargo extraído vs. cargo limpio, para cada destinatario real

Recorre los destinatarios GCBA ya cacheados en `extraction_cache_eval.json` (la
extracción real del LLM, no datos sinteticos) y muestra, lado a lado, el `cargo` tal
cual lo extrajo el modelo y el resultado de `clean_cargo` -- util para inspeccionar de
un vistazo si a algun cargo le quedo pegada la cadena institucional ("del Ministerio
de..."), que es exactamente el tipo de caso que encontramos con "titular".


In [ ]:
import sys

os.environ["RESOURCES_BASEPATH"] = str(Path("../../../resources").resolve())
for module_name in list(sys.modules):
    if module_name == "aymurai.settings" or module_name.startswith("aymurai.api"):
        del sys.modules[module_name]

from aymurai.api.endpoints.routers.llm.data_extraction import organigram_matching as om

EXTRACTION_CACHE_PATH_INSPECCION = Path(
    "../../../resources/data/restricted/defensoria/results/extraction_cache_eval.json"
).resolve()

cache_inspeccion = json.loads(EXTRACTION_CACHE_PATH_INSPECCION.read_text(encoding="utf-8"))

filas_clean_cargo = []
for doc_path, resultado in cache_inspeccion.items():
    for idx, dest in enumerate(resultado.get("destinatarios", [])):
        if dest.get("sector") != "GCBA" or not dest.get("cargo"):
            continue
        filas_clean_cargo.append({
            "doc_path": doc_path,
            "destinatario_idx": idx,
            "cargo_extraido": dest["cargo"],
            "cargo_limpio": om.clean_cargo(dest["cargo"]),
        })

clean_cargo_df = pd.DataFrame(filas_clean_cargo)
print(f"Destinatarios GCBA con cargo: {len(clean_cargo_df)}")
pd.set_option("display.max_colwidth", 120)
display(clean_cargo_df)


In [ ]:
clean_cargo_df[10:20]